In [1]:
#to use the language model, make sure you've unzipped the languageModel.tar.gz file
#and have compiled the code in the LanguageModelDecoder folder
baseDir = '/mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-main'

In [2]:
import os
from glob import glob
from pathlib import Path
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]=""

import numpy as np
from omegaconf import OmegaConf
import tensorflow as tf
from neuralDecoder.neuralSequenceDecoder import NeuralSequenceDecoder
import neuralDecoder.utils.lmDecoderUtils as lmDecoderUtils

2025-10-27 19:40:24.026299: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-27 19:40:26.614130: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libdirectml.d6f03b303ac3c4f2eeb8ca631688c9757b361310.so
2025-10-27 19:40:26.614200: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libdxcore.so
2025-10-27 19:40:26.618827: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libd3d12.so
Dropped Escape call with ulEscapeCode : 0x03007703
Dropped Escape call with ulEscapeCode : 0x03007703
2025-10-27 19:40:26.877628: I tensorflow/c/logging.cc:34] DirectML device enumeration: found 1 compatible adapters.


In [3]:
#loads the language model, could take a while and requires ~60 GB of memory
lmDir = baseDir+'/languageModel'
ngramDecoder = lmDecoderUtils.build_lm_decoder(
    lmDir,
    acoustic_scale=0.8, #1.2
    nbest=1,
    beam=18
)

I1027 19:40:33.149227   959 brain_speech_decoder.h:52] Reading fst /mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-main/languageModel/TLG.fst
I1027 19:55:41.355607   959 brain_speech_decoder.h:81] Reading symbol table /mnt/c/Users/krishna/Documents/NeuroSpeech/speechBCI-main/languageModel/words.txt


In [5]:
#evaluate the RNN on the test partition and competitionHoldOut partition
testDirs = ['test','competitionHoldOut']
trueTranscriptions = [[],[]]
decodedTranscriptions = [[],[]]
for dirIdx in range(2):
    ckptDir = baseDir + '/out/out'

    args = OmegaConf.load(os.path.join(ckptDir, 'args.yaml'))
    args['loadDir'] = ckptDir
    args['mode'] = 'infer'
    args['loadCheckpointIdx'] = None

    for x in range(len(args['dataset']['datasetProbabilityVal'])):
        args['dataset']['datasetProbabilityVal'][x] = 0.0

    for sessIdx in range(4,19):
        args['dataset']['datasetProbabilityVal'][sessIdx] = 1.0
        args['dataset']['dataDir'][sessIdx] = baseDir+'/derived/tfRecords'
    args['testDir'] = testDirs[dirIdx]

    # Initialize model
    tf.compat.v1.reset_default_graph()
    nsd = NeuralSequenceDecoder(args)

    # Inference
    out = nsd.inference()
    decoder_out = lmDecoderUtils.cer_with_lm_decoder(ngramDecoder, out, outputType='speech_sil', blankPenalty=np.log(2))

    def _ascii_to_text(text):
        endIdx = np.argwhere(text==0)
        return ''.join([chr(char) for char in text[0:endIdx[0,0]]])

    for x in range(out['transcriptions'].shape[0]):
        trueTranscriptions[dirIdx].append(_ascii_to_text(out['transcriptions'][x,:]))  
    decodedTranscriptions[dirIdx] = decoder_out['decoded_transcripts']


2025-10-27 20:20:17.714393: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-27 20:20:17.725893: I tensorflow/c/logging.cc:34] DirectML: creating device on adapter 0 (AMD Radeon RX 7900 XT)
Dropped Escape call with ulEscapeCode : 0x03007703
2025-10-27 20:20:18.407587: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-10-27 20:20:18.408933: W tensorflow/core/common_runtime/pluggable_device/pluggable_device_bfc_allocator.cc:28] Overriding allow_growth setting because force_memory_growth was requested by the device.
2025-10-27 20:20:18.410170: I tensorflow/core/c

/home/krishna/.local/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/krishna/.local/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer Orthogonal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "gru"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru_1 (GRU)                 multiple                  6292992   
                                                                 
 gru_2 (GRU)                 multiple                  1574400   
                                                                 
 gru_3 (GRU)                 multiple                  1574400   
                                                                 
 gru_4 (GRU)                 multiple                  1574400   
                                                                 
 gru_5 (GRU)                 multiple                  1574400   
                                                                 
 dense (Dense)               multiple                  21033     
                                                                 
Total params: 12,612,137
Trainable params: 12,612,137
Non-train

2025-10-27 20:20:25.699935: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 22020096 exceeds 10% of free system memory.
2025-10-27 20:20:26.373558: W tensorflow/core/framework/cpu_allocator_impl.cc:82] Allocation of 21381120 exceeds 10% of free system memory.
2025-10-27 20:20:26.492684: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-10-27 20:20:26.492735: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 33114 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)
2025-10-27 20:20:27.075967: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-10-27 20:20:27.244386: I tensorflow/core/comm

  0%|          | 0/600 [00:00<?, ?it/s]

Model: "gru"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru_1 (GRU)                 multiple                  6292992   
                                                                 
 gru_2 (GRU)                 multiple                  1574400   
                                                                 
 gru_3 (GRU)                 multiple                  1574400   
                                                                 
 gru_4 (GRU)                 multiple                  1574400   
                                                                 
 gru_5 (GRU)                 multiple                  1574400   
                                                                 
 dense (Dense)               multiple                  21033     
                                                                 
Total params: 12,612,137
Trainable params: 12,612,137
Non-train

2025-10-27 20:25:58.824713: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-10-27 20:25:58.824777: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 33114 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)
2025-10-27 20:25:58.835854: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-10-27 20:25:58.853393: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-10-27 20:25:58.853436: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_f

  0%|          | 0/1200 [00:00<?, ?it/s]

In [6]:
from neuralDecoder.utils.lmDecoderUtils import _cer_and_wer as cer_and_wer

#get word error rate and phoneme error rate for the test set (cer is actually phoneme error rate here)
cer, wer = cer_and_wer(decodedTranscriptions[0], trueTranscriptions[0], outputType='speech_sil', returnCI=True)

#print word error rate
print(wer)

(1.0002715915263445, 0.9991918049002733, 1.0016353341221642)


In [7]:
#print the sentence predictions for the test set
print(decodedTranscriptions[0])

['add your own caption', 'advertisement', 'loading', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'loading', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'ad

In [8]:
#print the predictions for the competition hold-out set (labels are unreleased)
print(decodedTranscriptions[1])

['advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'twitter', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'add your own caption', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'advertisement', 'twitter', 'adver

In [9]:
#format the predictions for competition submission. This generates a .txt file that can be submitted.
with open('baselineCompetitionSubmission.txt', 'w') as f:
    for x in range(len(decodedTranscriptions[1])):
        f.write(decodedTranscriptions[1][x]+'\n')